# Iniciación de paquetes

In [5]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np

# Métricas iniciales

In [6]:
import math

def metricas_estaticas(clientes, horas, mu, s, cs=0, cq=0):
   
    lambda_tasa = clientes / horas
    rho = lambda_tasa / (s * mu)
    

    if rho >= 1:
        return {"error": "Sistema inestable (rho >= 1)"}


    termino_sumatorio = sum([(lambda_tasa/mu)**n / math.factorial(n) for n in range(s)])
    termino_cola = ((lambda_tasa/mu)**s / (math.factorial(s) * (1 - rho)))
    pi0 = 1 / (termino_sumatorio + termino_cola)
    
  
    Lq = (pi0 * ((lambda_tasa/mu)**s) * rho) / (math.factorial(s) * (1 - rho)**2)
    Wq = Lq / lambda_tasa
    W = Wq + (1/mu)
    L = lambda_tasa * W
    
    
    Cts = (cs * s) + (cq * Lq)
    
    # Devolvemos un diccionario con todo
    return {
        "rho": round(rho, 3),
        "Lq": round(Lq, 3),
        "Wq_min": round(Wq*60, 3),
        "L": round(L, 3),
        "Cts": round(Cts, 3),
        "lambda": round(lambda_tasa, 3),
        "horas": horas,
        "clientes": clientes,
        "mu": mu,
        "servidores": s
    }

Aún no nos interesa la optimización del dinero, de momento vamos a fijarnos en los factores de utilización y ocupacion de las colas

In [9]:
metricas_teoricas = metricas_estaticas(120, 8, 3, 7)
print(metricas_teoricas)
metricas_teoricas6 = metricas_estaticas(120, 8, 3, 6)
metricas_teoricas5 = metricas_estaticas(120, 8, 3, 5)
print(metricas_teoricas6)
print(metricas_teoricas5)

{'rho': 0.714, 'Lq': 0.81, 'Wq_min': 3.241, 'L': 5.81, 'Cts': 0.0, 'lambda': 15.0, 'horas': 8, 'clientes': 120, 'mu': 3, 'servidores': 7}
{'rho': 0.833, 'Lq': 2.938, 'Wq_min': 11.75, 'L': 7.938, 'Cts': 0.0, 'lambda': 15.0, 'horas': 8, 'clientes': 120, 'mu': 3, 'servidores': 6}
{'error': 'Sistema inestable (rho >= 1)'}


# Función de simulación de llegadas y salidas

In [ ]:
import numpy as np
import pandas as pd

def generacion_llegadas(lambda_base, pesos_horarios):
    
    tiempos = []
    for hora, peso in enumerate(pesos_horarios):
        tasa_actual = lambda_base * peso
        tiempo_actual = float(hora)
        while True:
            inter_llegada = np.random.exponential(1 / tasa_actual)
            tiempo_actual += inter_llegada
            if tiempo_actual > hora + 1: break
            tiempos.append(tiempo_actual)
    return tiempos

def simulacion_atencion(asesores, llegadas):
    
    disponibilidad = [0.0] * len(asesores)
    esperas = []
    for llegada in llegadas:
        libre = min(disponibilidad)
        idx = disponibilidad.index(libre)
        inicio = max(llegada, libre)
        esperas.append(inicio - llegada)
        # El mu depende del asesor específico asignado
        t_serv = np.random.exponential(1 / asesores[idx]['mu'])
        disponibilidad[idx] = inicio + t_serv
    return esperas

def simulacion_total(n_iteraciones, lambda_base, pesos, lista_asesores, coste_espera_h):
    
    record_wq = []
    record_costes_espera = []
    
    # Coste fijo de los salarios (Suma de costes/h * 8 horas)
    coste_fijo_asesores = sum([a['coste'] for a in lista_asesores]) * len(pesos)
    
    for _ in range(n_iteraciones):
        llegadas = generacion_llegadas(lambda_base, pesos)
        esperas = simulacion_atencion(lista_asesores, llegadas)
        
        if esperas:
            wq_dia = np.mean(esperas)
            coste_espera_dia = sum(esperas) * coste_espera_h
            record_wq.append(wq_dia * 60) # Pasamos a minutos
            record_costes_espera.append(coste_espera_dia)

    # Consolidación Estadística
    return {
        "Wq Medio (min)": round(np.mean(record_wq), 2),
        "Incertidumbre (Std Dev min)": round(np.std(record_wq), 2),
        "Coste Salarios (€/día)": coste_fijo_asesores,
        "Coste Espera Medio (€/día)": round(np.mean(record_costes_espera), 2),
        "Coste Total Medio (€/día)": round(coste_fijo_asesores + np.mean(record_costes_espera), 2),
        "Factor de utilización medio": round((lambda_base / ( * 3)), 3)
    }

# Configuración de equipos

In [ ]:
mi_equipo1 = [
    {'nombre': '1', 'mu': 3, 'coste': 20},
    {'nombre': '2', 'mu': 3, 'coste': 20},
    {'nombre': '3', 'mu': 3, 'coste': 20},
    {'nombre': '4', 'mu': 3, 'coste': 20},
    {'nombre': '5', 'mu': 3, 'coste': 20},
    {'nombre': '6', 'mu': 3, 'coste': 20},
    {'nombre': '7', 'mu': 3, 'coste': 20}
] # Equipo de 7 miembros con mu constante

mi_equipo2 = [
    {'nombre': '1', 'mu': 3, 'coste': 20},
    {'nombre': '2', 'mu': 3, 'coste': 20},
    {'nombre': '3', 'mu': 3, 'coste': 20},
    {'nombre': '4', 'mu': 3, 'coste': 20},
    {'nombre': '5', 'mu': 3, 'coste': 20},
    {'nombre': '6', 'mu': 3, 'coste': 20}
] # Equipo de 6 miembros con mu constante

mi_equipo3 = [
    {'nombre': '1', 'mu': 3, 'coste': 20},
    {'nombre': '2', 'mu': 3, 'coste': 20},
    {'nombre': '3', 'mu': 3, 'coste': 20},
    {'nombre': '4', 'mu': 3, 'coste': 20},
    {'nombre': '5', 'mu': 3, 'coste': 20}
] # Equipo de 5 miembros con mu constante

# Variables de resultados

In [38]:
resultados1 = simulacion_total(
    n_iteraciones=500, 
    lambda_base=15, 
    pesos=[1.3, 1.2, 0.9, 0.7, 0.8, 1.2, 1.1, 0.8], 
    lista_asesores=mi_equipo1, 
    coste_espera_h=10
)
resultados1

{'Wq Medio (min)': np.float64(3.37),
 'Incertidumbre (Std Dev min)': np.float64(3.82),
 'Coste Salarios (€/día)': 1120,
 'Coste Espera Medio (€/día)': np.float64(69.97),
 'Coste Total Medio (€/día)': np.float64(1189.97),
 'Factor de utilización medio': 0.833}

In [39]:
resultados2 = simulacion_total(
    n_iteraciones=500, 
    lambda_base=15, 
    pesos=[1.3, 1.2, 0.9, 0.7, 0.8, 1.2, 1.1, 0.8], 
    lista_asesores=mi_equipo2, 
    coste_espera_h=10
)
resultados2

{'Wq Medio (min)': np.float64(8.43),
 'Incertidumbre (Std Dev min)': np.float64(7.01),
 'Coste Salarios (€/día)': 960,
 'Coste Espera Medio (€/día)': np.float64(173.5),
 'Coste Total Medio (€/día)': np.float64(1133.5),
 'Factor de utilización medio': 0.833}

# Creación de dataframe

In [ ]:
metricas_csv = pd.DataFrame({
    "lambda=15, mu=3, s=7" : [resultados],
    "lambda=15, mu=3, s=6" : [7, 8, 9],
    "lambda=15, mu=3, s=5" : [10, 11, 12],
})
metricas_csv

,"lambda=15, mu=3, s=7","lambda=15, mu=3, s=6","lambda=15, mu=3, s=5"
0,4,7,10
1,5,8,11
2,6,9,12
